This notebook is used to run the HDOCK-Multimer (HDM) assembly pipeline, with optional integration of crosslinking restraints between subunits.  

By default, the notebook runs on an example [PDB: 6F0K](https://www.rcsb.org/structure/6F0K), but by uploading your own stoi.json, crosslinks.txt and .PDB files to Google Drive, you can predict your complex by changing the path in the "Run" cell, as instructed in the cell.

**Inputs:** \
 1.stoi.json - a JSON file recording stoichiometry and subunit information of the complex.\
 2.crosslinks.txt - a TXT file recording crosslinking distance restraints between subunits (for the 6F0K demo, the crosslinking restraints are simulated from the native structure to provide a controlled example; you can replace them with restraints from experimental data).\
 3.subcomponents/ - a folder containing AlphaFold-Multimer-predicted subcomponent files. \

**Outputs:** \
A series of .PDB files containing the structures of the full complex.

For more information, see [HDOCK-Multimer](https://github.com/xyao7/HDOCK-Multimer).

In [ ]:
#@title Download and install HDM (~2 min)
!pip -q install -U "git+https://github.com/openmm/pdbfixer.git"
!pip -q install py3Dmol
!echo Installed python dependencies

!echo Install HDM
!wget -qnc https://github.com/xyao7/HDOCK-Multimer/archive/refs/heads/main.zip -O HDM-master.zip 
!unzip -q HDM-master.zip 
!cd HDOCK-Multimer-main && bash setup.sh 
!echo HDM Installed Successfully!

import py3Dmol
import os

def view_pdb_color_by_chain(pdb_path: str):
  pdb_content = open(pdb_path, "r").read()
  view = py3Dmol.view(width=400, height=300)
  view.addModelsAsFrames(pdb_content)

  # Get the list of chains in the protein
  chains = {i[21] for i in pdb_content.split("\n") if len(i) > 21}

  # Assign a color to each chain
  colors = ["red", "blue", "green", "orange", "purple", "yellow", "pink", "brown", "black", "gray", "cyan", "magenta", "olive", "maroon", "navy", "teal", "gold", "silver", "crimson"]
  colors = 10 * colors

  # Set the style for each chain
  for i, chain in enumerate(chains):
      view.setStyle({'chain': chain}, {'cartoon': {'color': colors[i]}})

  view.zoomTo()
  view.show()

In [ ]:
#@title View example elements {run: "auto"}

#@markdown  Here we demonstrate the input used to create a model of [PDB: 6F0K](https://www.rcsb.org/structure/6F0K), which is composed of 7 different chains.

#@markdown  - stoi.json - Defines the stoichiometry and subunits information.

#@markdown  - crosslinks.txt - Defines the crosslinking restraints between subunits.

#@markdown  - pdb files - subcomponent models predicted by AlphaFold-Multimer (AFM) or other similar methods, saved in the `subcomponents/` folder.

#@markdown  You can view elements in the input by choosing them and running the cell.

element_to_view = "stoi.json" #@param ["stoi.json", "crosslinks.txt", "subcomponents/6F0K_ABC.pdb", "subcomponents/6F0K_ABD.pdb", "subcomponents/6F0K_ABE.pdb", "subcomponents/6F0K_ABF.pdb", "subcomponents/6F0K_ABG.pdb", "subcomponents/6F0K_ACD.pdb", "subcomponents/6F0K_ACE.pdb", "subcomponents/6F0K_ACF.pdb", "subcomponents/6F0K_ACG.pdb", "subcomponents/6F0K_ADE.pdb", "subcomponents/6F0K_ADF.pdb", "subcomponents/6F0K_ADG.pdb", "subcomponents/6F0K_AEF.pdb","subcomponents/6F0K_AEG.pdb","subcomponents/6F0K_AFG.pdb","subcomponents/6F0K_BCD.pdb", "subcomponents/6F0K_BCE.pdb", "subcomponents/6F0K_BCF.pdb", "subcomponents/6F0K_BCG.pdb","subcomponents/6F0K_BDE.pdb","subcomponents/6F0K_BDF.pdb", "subcomponents/6F0K_BDG.pdb", "subcomponents/6F0K_BEF.pdb","subcomponents/6F0K_BEG.pdb","subcomponents/6F0K_BFG.pdb", "subcomponents/6F0K_CDE.pdb", "subcomponents/6F0K_CDF.pdb", "subcomponents/6F0K_CDG.pdb","subcomponents/6F0K_CEF.pdb","subcomponents/6F0K_CEG.pdb","subcomponents/6F0K_CFG.pdb", "subcomponents/6F0K_DEF.pdb", "subcomponents/6F0K_DEG.pdb","subcomponents/6F0K_DFG.pdb", "subcomponents/6F0K_EFG.pdb"]

example_path = "/content/HDOCK-Multimer-main/crosslinks/examples/6F0K"

stoi_path = os.path.join(example_path, "stoi.json")
crosslink_path = os.path.join(examples, "crosslinks.txt")
if element_to_view == "stoi.json":
  print(open(stoi_path, 'r').read())
elif element_to_view == "crosslinks.txt":
  print(open(crosslink_path, 'r').read())
else:
  view_pdb_color_by_chain(oa.path.join(example_path, element_to_view))
  

In [ ]:
#@title Run HDM

#@markdown The folder `path_task/` should have:
#@markdown (1) a file named "stoi.json" recording the stoichiometry and subunit information
#@markdown (2) a file named "crosslinks.txt" recording the crosslinking restraints between subunits
#@markdown and (3) a folder named "subcomponents/" storing subcomponent structure files predicted by AFM or other similar methods.

#@markdown The results will be saved to a new folder named "results", under the folder `path_task/`.

import os 

path_task="/content/HDOCK-Multimer-main/crosslinks/examples/6F0K/" #@param {type:"string"}
max_results_number = "10" #@param[1, 5, 10, 20]

!bash HDOCK-Multimer-main/HDM_assemble_crosslinks.sh \
  -stoi "{path_task}/stoi.json" \
  -crosslink "{path_task}/crosslinks.txt" \
  -sub_dir "{path_task}/subcomponents/" \
  -nmax {max_results_number}


In [ ]:
#@title Display output 3D structure models {run: "auto"}
model_num = "1" #@param [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
model_num = int(model_num)

result_path = os.path.join(path_task, "results")
output_filenames = [i for i in os.listdir(result_path) if i.endswith(".pdb")]
assert model_num < len(output_filenames), f"Only have {len(output_filenames)} models"
output_path = os.path.join(result_path, f"model_{model_num}.pdb")

view_pdb_color_by_chain(output_path)